In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler

In [42]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())
display(test.head())

Train shape: (39073, 15)
Test shape: (9769, 14)


,id,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,37194,50,Private,Masters,14,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,35,United-States,<=50K
1,31094,28,Private,Some-college,10,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,40,United-States,<=50K
2,33815,29,Private,HS-grad,9,Married-civ-spouse,Other-service,Wife,White,Female,0,1902,35,United-States,>50K
3,14501,32,Private,HS-grad,9,Never-married,Adm-clerical,Not-in-family,White,Female,0,0,40,United-States,<=50K
4,23400,62,State-gov,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States,<=50K


,id,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
0,7763,28,Private,Some-college,10,Never-married,Other-service,Not-in-family,Black,Male,0,0,40,United-States
1,23882,62,NaN,HS-grad,9,Married-civ-spouse,NaN,Husband,White,Male,0,0,40,United-States
2,30508,46,Self-emp-not-inc,Bachelors,13,Never-married,Prof-specialty,Not-in-family,Black,Male,0,0,23,United-States
3,28912,39,Private,Prof-school,15,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,35,United-States
4,19485,21,State-gov,Some-college,10,Never-married,Other-service,Own-child,White,Female,0,0,25,United-States


### 기본 정보 확인

In [ ]:
print("===== Train Info =====")
train.info()

print("\n===== Test Info =====")
test.info()

## 결측치 확인

" ?" 형태의 결측치를 NaN으로 변환

In [ ]:
train = train.replace(" ?", np.nan)
test = test.replace(" ?", np.nan)

print("===== Train Missing Values =====")
print(train.isnull().sum())

print("\n===== Test Missing Values =====")
print(test.isnull().sum())

## 변수 타입 분류

In [ ]:
target_col = "income"

cat_cols = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

num_cols = [
    "age",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

## Class Distribution 확인

In [ ]:
print(train[target_col].value_counts())
print(train[target_col].value_counts(normalize=True))

sns.countplot(x=target_col, data=train)
plt.title("Class Distribution of Income")
plt.show()

## 수치형 변수 요약 및 시각화

In [ ]:
display(train[num_cols].describe())

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(train[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

## 범주형 변수 분포 확인

In [ ]:
for col in cat_cols:
    print(f"\n===== {col} =====")
    print(train[col].value_counts())

## 결측치 처리

범주형 변수: 최빈값으로 대체

수치형 변수: 중앙값으로 대체


In [ ]:
for col in cat_cols:
    mode_value = train[col].mode()[0]

    train[col] = train[col].fillna(mode_value)
    test[col] = test[col].fillna(mode_value)

for col in num_cols:
    median_value = train[col].median()

    train[col] = train[col].fillna(median_value)
    test[col] = test[col].fillna(median_value)

print("결측치 처리 후 train:")
print(train.isnull().sum())

print("\n결측치 처리 후 test:")
print(test.isnull().sum())

## Target Encoding

5만 이하 -> 0

5만 초과 -> 1

In [ ]:
train[target_col] = train[target_col].map({
    "<=50K": 0,
    ">50K": 1
})

print(train[target_col].value_counts())

## Categorical Encoding
Label Encoding 사용

train/test를 함께 fit해서 unseen category 문제 방지

In [ ]:
encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([train[col], test[col]], axis=0)

    le.fit(combined)

    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

    encoders[col] = le

display(train.head())

## Feature Engineering

capital_gain과 capital_loss를 합친 변수 생성

In [ ]:
train["capital_total"] = (
    train["capital_gain"]
    - train["capital_loss"]
)

test["capital_total"] = (
    test["capital_gain"]
    - test["capital_loss"]
)

num_cols.append("capital_total")

## Scaling

수치형 변수 표준화

In [ ]:
scaler = StandardScaler()

train[num_cols] = scaler.fit_transform(train[num_cols])
test[num_cols] = scaler.transform(test[num_cols])

display(train.head())

### 최종 데이터 분리

In [ ]:
X_train = train.drop(columns=[target_col])
y_train = train[target_col]

X_test = test.copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

## 전처리된 데이터 저장

In [ ]:
X_train.to_csv("processed_X_train.csv", index=False)
y_train.to_csv("processed_y_train.csv", index=False)
X_test.to_csv("processed_X_test.csv", index=False)

print("전처리 완료")
print("Saved files:")
print("- processed_X_train.csv")
print("- processed_y_train.csv")
print("- processed_X_test.csv")